<a href="https://colab.research.google.com/github/lucaaccomando/MachineLearningFinal/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chicago Crime Arrest Prediction
## Luca Accomando
## Spring 2026
## Data Mining — Classification Project

## Project workflow
This notebook follows an industry-style analytics workflow:

1. **Problem Framing & Data Acquisition**
2. **Exploratory Data Analysis (EDA) & Data Preparation**
3. **Model Development, Evaluation & Business Interpretation**

## GitHub + Colab workflow
1. Create a **new GitHub repository** for your project.
2. Upload this notebook to your repository.
3. In GitHub, open the notebook in **Google Colab**.
4. Commit changes to GitHub as you work.
5. Submit your GitHub repository link when requested.

## Project requirements
- Use a **classification dataset** ✅
- Use **Random Forest** as one of your main models ✅
- Use **Google Colab** ✅
- Include **visualization, preparation, modeling, and interpretation** ✅
- Explain results in a way a manager or stakeholder could understand ✅


In [1]:
# Basic libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# AutoViz
!pip install autoviz -q
from autoviz.AutoViz_Class import AutoViz_Class

# scikit-learn tools (Colab-friendly replacement for PyCaret)
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay, classification_report, cohen_kappa_score

# Models to compare
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

# Warnings
import warnings
warnings.filterwarnings('ignore')

# Evaluation
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.5/67.5 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.6/180.6 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 255.9/255.9 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 38.1 MB/s eta 0:00:00
Imported v0.1.905. Please call AutoViz in this sequence:
    AV = AutoViz_Class()
    %matplotlib inline
    dfte = AV.AutoViz(filename, sep=',', depVar='', dfte=None, header=0, verbose=1, lowess=False,
               chart_format='svg',max_rows_analyzed=150000,max_cols_analyzed=30, save_plot_dir=None)


# Deliverable 1: Problem Framing & Data Acquisition

## What problem are you trying to solve?

Each year, the Chicago Police Department records hundreds of thousands of crime incidents across the city. However, **not all reported crimes result in an arrest**. This project builds a binary classification model to predict whether a reported crime will result in an arrest (True/False), using features such as crime type, location, time of day, and district.

Understanding what drives arrest outcomes allows city officials, police leadership, and researchers to surface data-driven patterns in policing — and to evaluate where resources or policy adjustments may be needed.

## What is the target variable?

- **Column:** `arrest`
- **Type:** Boolean → Binary classification
- **Values:** `True` (arrest made) = 1, `False` (no arrest) = 0

## What organization, industry, or domain could use this model?

| Stakeholder | Use Case |
|---|---|
| Chicago Police Department | Identify crime types and districts with low arrest rates for resource review |
| City Policy Analysts | Study patterns in arrest outcomes across neighborhoods |
| Criminal Justice Researchers | Benchmark fairness across geography and crime category |
| Community Organizations | Understand gaps in public safety outcomes |

## Why does this problem matter?

Arrest rate is a key performance indicator in public safety. If certain crime types or locations consistently show low arrest rates, that may indicate resource gaps, patrol inefficiencies, or structural disparities. A predictive model helps quantify those patterns and supports better decision-making.

## Where did the dataset come from?

- **Source:** [Chicago Data Portal — Crimes 2001 to Present](https://data.cityofchicago.org/Public-Safety/Crimes-2001-to-Present/ijzp-q8t2)
- **Maintained by:** City of Chicago (updated regularly)
- **Size:** 8+ million rows; sampled for modeling efficiency
- **Features:** Crime type, location description, date/time, district, community area, latitude/longitude, domestic flag, and arrest outcome

## Why did you choose this dataset?

The Chicago Crime Dataset was selected because it has the richest ecosystem of any dataset reviewed:
- Widely used in Kaggle competitions, Medium tutorials, and academic projects (e.g., Notre Dame)
- Hosted on an official government open data portal — reliable and regularly updated
- Contains temporal, geospatial, and categorical features ideal for feature engineering
- The target variable (`arrest`) is clean, binary, and has direct real-world meaning
- Large enough (8M+ rows) to support robust modeling while sampling keeps compute manageable


## Data loading options

Choose **one** of the options below:
- load a CSV from GitHub
- upload a CSV into Colab
- read from a direct URL

Keep your original raw data file in your GitHub repository whenever possible.


In [2]:
# Load data from Chicago Data Portal (100k sample via Socrata API)
data_url = "https://data.cityofchicago.org/resource/ijzp-q8t2.csv?$limit=100000"
df = pd.read_csv(data_url)

# Option B: Load from your GitHub repo after uploading the CSV
# data_url = "https://raw.githubusercontent.com/yourusername/chicago-crime-arrest-prediction/main/data/crimes_sample.csv"
# df = pd.read_csv(data_url)

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 22 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    100000 non-null  int64  
 1   case_number           100000 non-null  object 
 2   date                  100000 non-null  object 
 3   block                 100000 non-null  object 
 4   iucr                  100000 non-null  object 
 5   primary_type          100000 non-null  object 
 6   description           100000 non-null  object 
 7   location_description  99505 non-null   object 
 8   arrest                100000 non-null  bool   
 9   domestic              100000 non-null  bool   
 10  beat                  100000 non-null  int64  
 11  district              100000 non-null  int64  
 12  ward                  100000 non-null  int64  
 13  community_area        99997 non-null   float64
 14  fbi_code              100000 non-null  object 
 15  x

In [3]:
df.head()

,id,case_number,date,block,iucr,primary_type,description,location_description,arrest,domestic,beat,district,ward,community_area,fbi_code,x_coordinate,y_coordinate,year,updated_on,latitude,longitude,location
0,14170994,JK222622,2026-04-16T00:00:00.000,027XX W THOMAS ST,1320,CRIMINAL DAMAGE,TO VEHICLE,STREET,False,False,1211,12,36,24.0,14,1157913.0,1907204.0,2026,2026-04-23T15:42:27.000,41.901121,-87.695415,"\n, \n(41.901120816, -87.695414649)"
1,14170017,JK221515,2026-04-16T00:00:00.000,005XX N MC CLURG CT,4387,OTHER OFFENSE,VIOLATE ORDER OF PROTECTION,APARTMENT,False,True,1834,18,2,8.0,26,1179097.0,1904100.0,2026,2026-04-23T15:42:27.000,41.892145,-87.617700,"\n, \n(41.892144688, -87.617699811)"
2,14172242,JK224081,2026-04-16T00:00:00.000,020XX W GIDDINGS ST,1365,CRIMINAL TRESPASS,TO RESIDENCE,RESIDENCE,False,False,1911,19,47,4.0,26,1162011.0,1931545.0,2026,2026-04-23T15:42:27.000,41.967829,-87.679681,"\n, \n(41.9678294, -87.679680678)"
3,14173513,JK225700,2026-04-16T00:00:00.000,012XX W ROSCOE ST,0710,THEFT,THEFT FROM MOTOR VEHICLE,STREET,False,False,1924,19,44,6.0,06,1167440.0,1922714.0,2026,2026-04-23T15:42:27.000,41.943481,-87.659974,"\n, \n(41.943481394, -87.659973917)"
4,14170646,JK220816,2026-04-16T00:00:00.000,034XX W 117TH ST,1242,DECEPTIVE PRACTICE,COMPUTER FRAUD,RESIDENCE,False,False,2211,22,19,74.0,11,1155525.0,1826811.0,2026,2026-04-23T15:42:27.000,41.680559,-87.706340,"\n, \n(41.680559288, -87.706339549)"


# Deliverable 2: Exploratory Data Analysis (EDA) & Data Preparation

## Goal
Understand and prepare the data for modeling. Assess data quality, identify patterns and anomalies, and engineer features that will help the model predict arrest outcomes.

## What to include
- Basic shape and structure of the data
- Variable types
- Missing values
- Class balance of the target
- Visualizations that help explain the data
- Preparation steps used before modeling

## Suggested questions to resolve
- Are there missing values?
- Are the classes balanced?
- Which variables might be useful predictors?
- Are any variables likely to cause problems?
- Do I need to eliminate any variables due to correlation, redundancy, or uniqueness (e.g., id columns)?


In [4]:
# Basic data inspection
print('Shape:', df.shape)
display(df.head())
display(df.info())
display(df.describe(include='all').T)


Shape: (100000, 22)


,id,case_number,date,block,iucr,primary_type,description,location_description,arrest,domestic,beat,district,ward,community_area,fbi_code,x_coordinate,y_coordinate,year,updated_on,latitude,longitude,location
0,14170994,JK222622,2026-04-16T00:00:00.000,027XX W THOMAS ST,1320,CRIMINAL DAMAGE,TO VEHICLE,STREET,False,False,1211,12,36,24.0,14,1157913.0,1907204.0,2026,2026-04-23T15:42:27.000,41.901121,-87.695415,"\n, \n(41.901120816, -87.695414649)"
1,14170017,JK221515,2026-04-16T00:00:00.000,005XX N MC CLURG CT,4387,OTHER OFFENSE,VIOLATE ORDER OF PROTECTION,APARTMENT,False,True,1834,18,2,8.0,26,1179097.0,1904100.0,2026,2026-04-23T15:42:27.000,41.892145,-87.617700,"\n, \n(41.892144688, -87.617699811)"
2,14172242,JK224081,2026-04-16T00:00:00.000,020XX W GIDDINGS ST,1365,CRIMINAL TRESPASS,TO RESIDENCE,RESIDENCE,False,False,1911,19,47,4.0,26,1162011.0,1931545.0,2026,2026-04-23T15:42:27.000,41.967829,-87.679681,"\n, \n(41.9678294, -87.679680678)"
3,14173513,JK225700,2026-04-16T00:00:00.000,012XX W ROSCOE ST,0710,THEFT,THEFT FROM MOTOR VEHICLE,STREET,False,False,1924,19,44,6.0,06,1167440.0,1922714.0,2026,2026-04-23T15:42:27.000,41.943481,-87.659974,"\n, \n(41.943481394, -87.659973917)"
4,14170646,JK220816,2026-04-16T00:00:00.000,034XX W 117TH ST,1242,DECEPTIVE PRACTICE,COMPUTER FRAUD,RESIDENCE,False,False,2211,22,19,74.0,11,1155525.0,1826811.0,2026,2026-04-23T15:42:27.000,41.680559,-87.706340,"\n, \n(41.680559288, -87.706339549)"


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 22 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    100000 non-null  int64  
 1   case_number           100000 non-null  object 
 2   date                  100000 non-null  object 
 3   block                 100000 non-null  object 
 4   iucr                  100000 non-null  object 
 5   primary_type          100000 non-null  object 
 6   description           100000 non-null  object 
 7   location_description  99505 non-null   object 
 8   arrest                100000 non-null  bool   
 9   domestic              100000 non-null  bool   
 10  beat                  100000 non-null  int64  
 11  district              100000 non-null  int64  
 12  ward                  100000 non-null  int64  
 13  community_area        99997 non-null   float64
 14  fbi_code              100000 non-null  object 
 15  x

None

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,100000.0,NaN,NaN,NaN,14066439.5245,591197.355451,28932.0,14051482.75,14091234.5,14131004.5,14175468.0
case_number,100000,99993,JJ482411,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date,100000,53741,2026-01-01T00:00:00.000,57,NaN,NaN,NaN,NaN,NaN,NaN,NaN
block,100000,22322,0000X N STATE ST,253,NaN,NaN,NaN,NaN,NaN,NaN,NaN
iucr,100000,306,0486,8259,NaN,NaN,NaN,NaN,NaN,NaN,NaN
primary_type,100000,30,THEFT,22389,NaN,NaN,NaN,NaN,NaN,NaN,NaN
description,100000,286,SIMPLE,12429,NaN,NaN,NaN,NaN,NaN,NaN,NaN
location_description,99505,114,STREET,27007,NaN,NaN,NaN,NaN,NaN,NaN,NaN
arrest,100000,2,False,84621,NaN,NaN,NaN,NaN,NaN,NaN,NaN
domestic,100000,2,False,80557,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# Missing values summary
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]
print('Columns with missing values:')
display(missing_df)

# Visualize missing values
if not missing_df.empty:
    plt.figure(figsize=(10, 4))
    sns.barplot(x=missing_df['Missing %'], y=missing_df.index, palette='Reds_d')
    plt.title('Missing Value Percentage by Column')
    plt.xlabel('Missing (%)')
    plt.tight_layout()
    plt.show()

print('\nDecision: Missing values are minimal (<1%). Categorical columns will be imputed with mode, numeric with median.')


Columns with missing values:


,Missing Count,Missing %
location_description,495,0.50
x_coordinate,76,0.08
location,76,0.08
y_coordinate,76,0.08
longitude,76,0.08
latitude,76,0.08
community_area,3,0.00



Decision: Missing values are minimal (<1%). Categorical columns will be imputed with mode, numeric with median.


In [6]:
target = 'arrest'

# Class balance
arrest_counts = df[target].value_counts()
arrest_pct = df[target].value_counts(normalize=True) * 100
display(pd.DataFrame({'Count': arrest_counts, 'Percent': arrest_pct.round(1)}))

# Plot class balance
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['No Arrest (False)', 'Arrest (True)'], arrest_counts.values,
       color=['#d9534f', '#5cb85c'], edgecolor='white')
ax.set_title('Target Variable: Arrest Outcome Distribution')
ax.set_ylabel('Number of Incidents')
for i, v in enumerate(arrest_counts.values):
    ax.text(i, v + 500, f'{arrest_pct.values[i]:.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

print('Insight: Classes are imbalanced. Stratified split and class_weight will be used.')


,Count,Percent
arrest,,
False,84621,84.6
True,15379,15.4


Insight: Classes are imbalanced. Stratified split and class_weight will be used.


## AutoViz integration

AutoViz is useful for fast exploratory analysis. It can generate many plots at once.

**Important for Colab:** after AutoViz runs, use `plt.close('all')` before creating your own plots later in the notebook. This helps prevent old figures from appearing unexpectedly.


In [7]:
# Install and run AutoViz in Colab if needed
# You may comment this out if AutoViz is already installed in your runtime

# !pip install autoviz

from autoviz.AutoViz_Class import AutoViz_Class
AV = AutoViz_Class()


In [8]:
# Run AutoViz on a 10k sample to keep it fast
df_av = df.sample(10000, random_state=42)

AV.AutoViz(
    '',
    sep=',',
    depVar=target,
    dfte=df_av,
    header=0,
    verbose=1,
    lowess=False,
    chart_format='svg',
    max_rows_analyzed=10000,
    max_cols_analyzed=20
)

import matplotlib.pyplot as plt
plt.close('all')


Shape of your Data Set loaded: (10000, 22)
#######################################################################################
######################## C L A S S I F Y I N G  V A R I A B L E S  ####################
#######################################################################################
Classifying variables in data set...
    Number of Numeric Columns =  5
    Number of Integer-Categorical Columns =  3
    Number of String-Categorical Columns =  4
    Number of Factor-Categorical Columns =  0
    Number of String-Boolean Columns =  1
    Number of Numeric-Boolean Columns =  1
    Number of Discrete String Columns =  2
    Number of NLP String Columns =  3
    Number of Date Time Columns =  0
    Number of ID Columns =  2
    Number of Columns to Delete =  0
    21 Predictors classified...
        2 variable(s) removed since they were ID or low-information variables
        List of variables removed: ['id', 'case_number']
Since Number of Rows in data 10000 exceeds ma

,Data Type,Missing Values%,Unique Values%,Minimum Value,Maximum Value,DQ Issue
date,object,0.000000,86,,,No issue
block,object,0.000000,66,,,No issue
iucr,object,0.000000,2,,,Possible high cardinality column with 228 unique values: Use hash encoding or text embedding to reduce dimension.
primary_type,object,0.000000,0,,,17 rare categories: Too many to list. Group them into a single category or drop the categories.
description,object,0.000000,2,,,Possible high cardinality column with 214 unique values: Use hash encoding or text embedding to reduce dimension.
location_description,object,0.410123,0,,,"41 missing values. Impute them with mean, median, mode, or a constant value such as 123., 79 rare categories: Too many to list. Group them into a single category or drop the categories., Mixed dtypes: has 2 different data types: object, float,"
domestic,bool,0.000000,0,0.000000,1.000000,No issue
beat,int64,0.000000,2,111.000000,2535.000000,No issue
district,int64,0.000000,0,1.000000,25.000000,Column has a high correlation with ['beat']. Consider dropping one of them.
ward,int64,0.000000,0,1.000000,50.000000,No issue


Total Number of Scatter Plots = 15


[nltk_data] Downloading collection 'popular'
[nltk_data]    | 
[nltk_data]    | Downloading package cmudict to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/cmudict.zip.
[nltk_data]    | Downloading package gazetteers to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/gazetteers.zip.
[nltk_data]    | Downloading package genesis to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/genesis.zip.
[nltk_data]    | Downloading package gutenberg to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/gutenberg.zip.
[nltk_data]    | Downloading package inaugural to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/inaugural.zip.
[nltk_data]    | Downloading package movie_reviews to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping corpora/movie_reviews.zip.
[nltk_data]    | Downloading package names to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/names.zip.
[nltk_data]    | Downloading package shakespeare to /root/nltk_data...
[nlt

Could not draw wordcloud plot for date. 
Looks like you are missing some required data for this feature.

To download the necessary data, simply run

    python -m textblob.download_corpora

or use the NLTK downloader to download the missing data: http://nltk.org/data.html
If this doesn't fix the problem, file an issue at https://github.com/sloria/TextBlob/issues.

Could not draw wordcloud plot for block. 
Looks like you are missing some required data for this feature.

To download the necessary data, simply run

    python -m textblob.download_corpora

or use the NLTK downloader to download the missing data: http://nltk.org/data.html
If this doesn't fix the problem, file an issue at https://github.com/sloria/TextBlob/issues.

Could not draw wordcloud plot for location. 
Looks like you are missing some required data for this feature.

To download the necessary data, simply run

    python -m textblob.download_corpora

or use the NLTK downloader to download the missing data: http://nltk

In [9]:
# --- Feature Engineering: extract time components from date ---
df['date'] = pd.to_datetime(df['date'])
df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month

# --- Chart 1: Arrest Rate by Primary Crime Type ---
arrest_by_type = (
    df.groupby('primary_type')[target].mean() * 100
).sort_values(ascending=False).reset_index()
arrest_by_type.columns = ['primary_type', 'arrest_rate']

plt.figure(figsize=(14, 8))
sns.barplot(data=arrest_by_type, x='arrest_rate', y='primary_type', palette='Blues_d')
plt.title('Arrest Rate by Primary Crime Type (%)')
plt.xlabel('Arrest Rate (%)')
plt.ylabel('Crime Type')
plt.tight_layout()
plt.show()
print('Insight: Crime type is a strong predictor — narcotics and weapons show high arrest rates, theft shows low.')

# --- Chart 2: Arrest Rate by Hour of Day ---
arrest_by_hour = df.groupby('hour')[target].mean() * 100

plt.figure(figsize=(12, 5))
arrest_by_hour.plot(kind='line', marker='o', color='steelblue', linewidth=2)
plt.title('Arrest Rate by Hour of Day')
plt.xlabel('Hour (0 = Midnight, 12 = Noon)')
plt.ylabel('Arrest Rate (%)')
plt.xticks(range(0, 24))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('Insight: Arrest rates vary by hour — making hour a useful engineered feature.')

# --- Chart 3: Arrest Rate by Day of Week ---
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
arrest_by_day = df.groupby('day_of_week')[target].mean() * 100
arrest_by_day.index = day_names

plt.figure(figsize=(9, 4))
arrest_by_day.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Arrest Rate by Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Arrest Rate (%)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# --- Chart 4: Domestic vs Non-Domestic ---
arrest_by_domestic = df.groupby('domestic')[target].mean() * 100
arrest_by_domestic.index = ['Non-Domestic', 'Domestic']

plt.figure(figsize=(6, 4))
arrest_by_domestic.plot(kind='bar', color=['#5b9bd5', '#ed7d31'], edgecolor='white')
plt.title('Arrest Rate: Domestic vs Non-Domestic Incidents')
plt.ylabel('Arrest Rate (%)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
print('Insight: Domestic incidents show a different arrest rate pattern — include domestic as a feature.')

# --- Chart 5: Top Location Types by Arrest Rate ---
location_arrest = (
    df.groupby('location_description')[target]
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'arrest_rate', 'count': 'incidents'})
)
location_arrest = location_arrest[location_arrest['incidents'] >= 200]
location_arrest['arrest_rate'] *= 100
top_locations = location_arrest.sort_values('arrest_rate', ascending=False).head(10)

plt.figure(figsize=(12, 5))
sns.barplot(x=top_locations['arrest_rate'], y=top_locations.index, palette='Greens_d')
plt.title('Top 10 Location Types by Arrest Rate (min. 200 incidents)')
plt.xlabel('Arrest Rate (%)')
plt.ylabel('Location Type')
plt.tight_layout()
plt.show()


Insight: Crime type is a strong predictor — narcotics and weapons show high arrest rates, theft shows low.
Insight: Arrest rates vary by hour — making hour a useful engineered feature.
Insight: Domestic incidents show a different arrest rate pattern — include domestic as a feature.


## Data preparation plan

Explain your preparation steps in plain language:
- columns dropped
- missing value handling
- encoding categorical variables
- train/test split strategy
- any feature engineering

Write a short summary in the markdown cell below this one.


### Student Preparation Summary

**Features selected based on EDA:**
- `primary_type` — strongest predictor of arrest outcome
- `location_description` — location type influences arrest likelihood
- `district` — geographic grouping of police coverage
- `domestic` — converted from bool to int; domestic incidents show different arrest patterns
- `hour`, `day_of_week`, `month` — engineered from the `date` column; time of incident matters

**Columns dropped:**
- `id`, `case_number`, `block`, `iucr`, `fbi_code` — identifiers or redundant codes, not predictive
- `latitude`, `longitude`, `x_coordinate`, `y_coordinate`, `location` — not used as direct model features
- `updated_on`, `year` — not relevant to predicting arrest outcome

**Missing value handling:**
- Categorical columns: imputed with most frequent value (mode)
- Numeric columns: imputed with median
- Missing values are minimal (<1%) so no rows were dropped

**Encoding:**
- `primary_type` and `location_description` encoded with OneHotEncoder
- `domestic` converted to integer (0/1) before pipeline

**Train/test split:**
- 80% train / 20% test
- Stratified by target to preserve class balance in both sets


# Deliverable 3: Model Development, Evaluation & Interpretation

## What to include
- preprocessing pipeline
- Random Forest model
- parameter tuning
- evaluation on the test set
- confusion matrix
- kappa
- feature importance
- interpretation of what the results mean

## Reminder
You should explain results in a business-friendly way, not only with technical language.


In [10]:
# Modeling imports
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    cohen_kappa_score
)


In [11]:
# Select features based on EDA findings
# Convert domestic bool to int to avoid dtype issues with SimpleImputer
df['domestic'] = df['domestic'].astype(int)

features = ['primary_type', 'location_description', 'district', 'domestic', 'hour', 'day_of_week', 'month']
df_model = df[features + [target]].dropna().copy()
df_model[target] = df_model[target].astype(int)

X = df_model[features]
y = df_model[target]

categorical_cols = ['primary_type', 'location_description']
numeric_cols = ['district', 'domestic', 'hour', 'day_of_week', 'month']

print('Categorical columns:', categorical_cols)
print('Numeric columns:', numeric_cols)
print('Dataset size:', X.shape)
print('\nTarget distribution:')
print(y.value_counts(normalize=True).round(3))


Categorical columns: ['primary_type', 'location_description']
Numeric columns: ['district', 'domestic', 'hour', 'day_of_week', 'month']
Dataset size: (99505, 7)

Target distribution:
arrest
0    0.845
1    0.155
Name: proportion, dtype: float64


In [12]:
# Build preprocessing pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

# Train/test split — stratified to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=123, stratify=y
)

print(f'Training set: {X_train.shape[0]} rows')
print(f'Test set:     {X_test.shape[0]} rows')


Training set: 79604 rows
Test set:     19901 rows


In [13]:
# Baseline Random Forest model
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=123))
])

rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred))

print("Cohen's Kappa:", round(cohen_kappa_score(y_test, y_pred), 4))


Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.97      0.93     16825
           1       0.70      0.42      0.52      3076

    accuracy                           0.88     19901
   macro avg       0.80      0.69      0.73     19901
weighted avg       0.87      0.88      0.87     19901

Cohen's Kappa: 0.4606


In [14]:
# Confusion matrix
import matplotlib.pyplot as plt

plt.close('all')
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(ax=ax)
ax.set_title("Random Forest Confusion Matrix")
plt.show()


## Hyperparameter tuning

We do not know the best settings ahead of time, so we try multiple combinations.

A parameter grid gives the model several choices for each setting. GridSearchCV tests combinations and selects the version that performs best according to the scoring metric.


In [15]:
# Tune the Random Forest model
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 5, 10],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

rf_tuning_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=123))
])

grid_search = GridSearchCV(
    estimator=rf_tuning_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="f1_weighted",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
best_rf = grid_search.best_estimator_

print("Best Parameters:", grid_search.best_params_)


Best Parameters: {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__min_samples_split': 5, 'model__n_estimators': 200}


In [16]:
# Final evaluation on the test set
best_preds = best_rf.predict(X_test)

print("Tuned Random Forest Classification Report:")
print(classification_report(y_test, best_preds))

kappa = cohen_kappa_score(y_test, best_preds)
print("Cohen's Kappa:", round(kappa, 4))


Tuned Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.98      0.94     16825
           1       0.78      0.40      0.53      3076

    accuracy                           0.89     19901
   macro avg       0.84      0.69      0.73     19901
weighted avg       0.88      0.89      0.87     19901

Cohen's Kappa: 0.4712


In [17]:
# Tuned confusion matrix
plt.close('all')
cm = confusion_matrix(y_test, best_preds)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(ax=ax)
ax.set_title("Tuned Random Forest Confusion Matrix")
plt.show()


### Reading the Confusion Matrix

The confusion matrix shows how many predictions the model got right and wrong for each class:

| | **Predicted: No Arrest (0)** | **Predicted: Arrest (1)** |
|---|---|---|
| **Actual: No Arrest (0)** | True Negatives (TN) — correctly predicted no arrest | False Positives (FP) — predicted arrest, but no arrest occurred |
| **Actual: Arrest (1)** | False Negatives (FN) — predicted no arrest, but arrest did occur | True Positives (TP) — correctly predicted arrest |

**Key takeaway for this model:**
- The model is very good at identifying non-arrest incidents (high TN count).
- It struggles with actual arrest cases — the large FN count means many real arrests are predicted as non-arrests. This is a direct consequence of class imbalance: only ~15% of incidents result in an arrest, so the model has far fewer examples to learn from.
- False Positives are relatively low, meaning when the model does predict an arrest, it is usually correct.
- **Business implication:** Relying on this model alone to flag "likely arrest" cases would miss the majority of actual arrests. It performs best as a tool for identifying cases very unlikely to result in arrest, rather than confidently predicting which ones will.

## Feature importance

Feature importance helps us see which inputs influenced the Random Forest most.

Be careful:
- importance does **not** prove causation
- importance can be split across multiple one-hot encoded columns
- importance tells us what mattered to the model, not necessarily what matters in the real world


In [18]:
# Feature importance from the tuned model
import pandas as pd

feature_names = best_rf.named_steps["preprocessor"].get_feature_names_out()
importances = best_rf.named_steps["model"].feature_importances_

feature_importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

display(feature_importance_df.head(15))


,feature,importance
22,cat__primary_type_NARCOTICS,0.170795
0,num__district,0.148905
2,num__hour,0.147035
3,num__day_of_week,0.082635
4,num__month,0.074417
33,cat__primary_type_WEAPONS VIOLATION,0.070242
17,cat__primary_type_INTERFERENCE WITH PUBLIC OFFICER,0.022689
32,cat__primary_type_THEFT,0.014435
7,cat__primary_type_BATTERY,0.012987
10,cat__primary_type_CRIMINAL DAMAGE,0.012355


In [19]:
# Plot top feature importances
top_n = 15
top_features = feature_importance_df.head(top_n).sort_values("importance")

plt.figure(figsize=(8, 6))
plt.barh(top_features["feature"], top_features["importance"])
plt.title(f"Top {top_n} Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()


## Interpretation prompts

Write short answers below:
- How well did the model perform? The tuned Random Forest achieved an overall accuracy of 89% with a Cohen's Kappa of 0.48. Kappa measures agreement beyond random chance — a score of 0.48 indicates moderate agreement, meaning the model is doing meaningfully better than simply guessing the majority class every time. Tuning improved both accuracy and Kappa slightly over the baseline model (88% accuracy, Kappa 0.457), with the best configuration using 200 trees and no depth limit.
- Which class was easier or harder to predict? Non-arrest incidents (class 0) were much easier to predict — the model achieved 98% recall and a 0.94 F1 score for that class. Arrest incidents (class 1) were significantly harder, with only 40% recall and a 0.53 F1 score. This means the model correctly identifies most non-arrests but misses roughly 6 out of 10 actual arrests. This is largely explained by the class imbalance — only 15.4% of incidents in the dataset resulted in an arrest, giving the model far fewer examples to learn from.
- Which variables seemed most important? Based on the feature importance chart, crime type (primary_type) was the dominant predictor, consistent with our EDA finding that narcotics and weapons violations have dramatically higher arrest rates than theft or criminal damage. Time-based features (hour, day_of_week) and location type (location_description) also ranked among the top contributors, confirming that when and where a crime occurs meaningfully influences arrest likelihood.
- Where did the model make mistakes? The confusion matrix reveals that the model's main errors are False Negatives — cases where an arrest actually occurred but the model predicted no arrest. This is the most costly error type in practice, since it means the model misses roughly 6 out of 10 actual arrests. False Positives (predicted arrest but no arrest occurred) are far less common, meaning the model is conservative about flagging arrests. The root cause is class imbalance: with only ~15% of incidents resulting in an arrest, the model learns a bias toward predicting the majority class (no arrest). Applying class_weight='balanced' or resampling techniques such as SMOTE would be the most direct fix.
- How could this model be used by a real organization? A city analyst or police department could use this model to flag crime types and locations that are statistically unlikely to result in an arrest, then investigate whether those patterns reflect resource shortages, patrol gaps, or systemic issues. For example, if certain districts consistently show low predicted arrest rates for a given crime type, leadership could use that insight to reallocate resources or adjust patrol strategy. The model is best used as a decision-support tool — not as a replacement for human judgment.
- What would you improve next? Address class imbalance using SMOTE or class_weight='balanced' in the Random Forest
Add more features such as community_area and beat for finer geographic resolution
Test additional models such as XGBoost which often handles imbalanced data better
Expand the sample size beyond 100k rows to improve generalization
Evaluate using precision-recall AUC rather than accuracy, which is more informative for imbalanced datasets


### Student interpretation summary
Replace this section with your final written interpretation.


# Optional: Save your final processed data file and model

You may save your trained model if you want to show a deployment-style step.


In [20]:
import joblib

# Example:
# joblib.dump(best_rf, "final_model.pkl")
# print("Model saved.")

# saving data file
from google.colab import drive
drive.mount('/content/drive')

# Save to Drive
df_clean.to_csv('/content/drive/MyDrive/cleaned_data.csv', index=False)

MessageError: Error: credential propagation was unsuccessful